# OVD-Diagnose — cross-domain OVD failure benchmark

Frozen OVD detectors in Global / Oracle / Agnostic modes → (L, S, C) fingerprint, with
detector-intrinsic `L_det`, bootstrap CIs, and synthetic-control validation. See `paper/DESIGN.md`.

**Headless-ready:** Save Version → Save & Run All (Commit). GPU + Internet on.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true

In [ ]:
# sentence-transformers -> semantic vocab control
!pip install -q ultralytics transformers pycocotools pyyaml sentence-transformers
!python tools/ovd_diagnose.py   # metric self-test

## Aerial domain (LAE-80C) — download + recursive flatten

In [ ]:
import os, zipfile, shutil, json
BASE = '/kaggle/working/YOLOBERT/data/aerial'
ANN_DIR, IMG_DIR = f'{BASE}/annotations', f'{BASE}/images'
os.makedirs(ANN_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
ann = f'{ANN_DIR}/instances_val.json'
HF = 'https://huggingface.co/datasets/jaychempan/LAE-1M/resolve/main/LAE-80C'
if not os.path.exists(ann):
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.json?download=true" -O {ann}
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark_categories.json?download=true" -O {ANN_DIR}/categories.json
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.txt?download=true" -O {ANN_DIR}/classes.txt
def count_jpgs():
    return sum(1 for _, _, fs in os.walk(IMG_DIR) for f in fs if f.lower().endswith('.jpg'))
n_expected = len(json.load(open(ann))['images'])
if count_jpgs() < n_expected:
    zp = '/kaggle/working/images.zip'
    if not os.path.exists(zp) or os.path.getsize(zp) < 5e8:
        !wget -q --no-check-certificate "{HF}/images.zip?download=true" -O {zp}
    with zipfile.ZipFile(zp) as z: z.extractall(IMG_DIR)
    if os.path.exists(zp): os.remove(zp)
for root, _, files in os.walk(IMG_DIR):
    if root == IMG_DIR: continue
    for f in files:
        if f.lower().endswith('.jpg'):
            dst = os.path.join(IMG_DIR, f)
            if not os.path.exists(dst): shutil.move(os.path.join(root, f), dst)
for root, dirs, files in os.walk(IMG_DIR, topdown=False):
    if root != IMG_DIR and not os.listdir(root): os.rmdir(root)
print('images:', len([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.jpg')]),
      '| expected:', n_expected)

## Run all models  (`LIMIT=0` full; `200` quick). Writes fingerprints + per-model results.

In [ ]:
LIMIT = 0
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/aerial/annotations/instances_val.json \
  --imgs data/aerial/images \
  --out  runs/diag/aerial \
  --device cuda:0 {limit_arg} \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
# Fingerprint table — now includes L_det (detector-intrinsic localization, no SAM)
import pandas as pd
print(pd.read_csv('runs/diag/aerial/fingerprints.csv').to_string(index=False))

## Uncertainty — bootstrap 95% CIs on the IoA-F1 anchor (per model)

In [ ]:
import json
from tools.ovd_diagnose import bootstrap_ci
ann = 'data/aerial/annotations/instances_val.json'
for m in ['yoloworld', 'owlv2', 'groundingdino']:
    p = f'runs/diag/aerial/{m}/results_global.json'
    if os.path.exists(p):
        ci = bootstrap_ci(ann, json.load(open(p)), metric='ioa_f1', n_boot=1000)
        print(f"{m:>14}: IoA-F1 = {ci['mean']:.4f}  [{ci['lo']:.4f}, {ci['hi']:.4f}]")

## Validation — synthetic controls
Each control perturbs one factor; only its axis should move (construct validity).

In [ ]:
# Temperature (post-hoc): C_ece must change while AP (hence S) stays flat.
!python tools/synthetic_controls.py --control temperature \
  --ann data/aerial/annotations/instances_val.json \
  --results runs/diag/aerial/owlv2/results_global.json \
  --out runs/controls/aerial_owlv2_temperature.csv

In [ ]:
# Vocabulary control: RANDOM vs SEMANTIC distractors. If semantic hurts MORE than random,
# S measures semantic confusion, not just vocabulary size.
for mode in ['random', 'semantic']:
    !python tools/synthetic_controls.py --control vocab --distractor {mode} \
      --ann data/aerial/annotations/instances_val.json --imgs data/aerial/images \
      --model owlv2 --weights google/owlv2-base-patch16-ensemble \
      --out runs/controls/aerial_vocab_{mode}.csv --limit 300 --device cuda:0

In [ ]:
# Blur control: L must rise as inputs are blurred (text-independent).
!python tools/synthetic_controls.py --control blur \
  --ann data/aerial/annotations/instances_val.json --imgs data/aerial/images \
  --sam_weights mobile_sam.pt --out runs/controls/aerial_blur.csv --limit 200 --device cuda:0

In [ ]:
import pandas as pd, os
for name in ['aerial_owlv2_temperature', 'aerial_vocab_random', 'aerial_vocab_semantic', 'aerial_blur']:
    p = f'runs/controls/{name}.csv'
    if os.path.exists(p):
        print(f'== {name} =='); print(pd.read_csv(p).to_string(index=False), '\n')

**Expected:** temperature → `C_ece` varies, `AP` flat; vocab → `S` rises, and **semantic ≥ random**
at each distractor count; blur → `L` rises. Together with `L_det` agreeing with SAM-based `L` and
the bootstrap CIs, these address the main reviewer concerns (axis identifiability, S validity, uncertainty).